# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR²](https://sen.science/doi/10.71728/senscience.vq4a-28xa) dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All entities (record sets, fields, columns) are referenced by their unique `@id`.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata fields
print(f"Dataset title: {dataset.metadata.name}")
print(f"Description: {dataset.metadata.description}\n")
print(f"Published: {getattr(dataset.metadata, 'datePublished', 'N/A')}")
print(f"Version: {getattr(dataset.metadata, 'version', 'N/A')}\n")
print(f"Keywords: {getattr(dataset.metadata, 'keywords', 'N/A')}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

In [ ]:
# List all record sets and their @ids
record_sets = dataset.record_sets
print(f"Total record sets found: {len(record_sets)}\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    print(f"  name: {rs.get('name', 'N/A')}")
    fields = rs.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    print(f"  Fields/Columns (@ids):")
    for field in fields:
        if isinstance(field, dict):
            print(f"    - {field.get('@id', field)}")
        else:
            print(f"    - {field}")
    print()

## 3. Data Extraction
Load data from **each record set** into a DataFrame for analysis. Use the `@id` values found above.

In [ ]:
# Replace with the specific record set @ids identified above
record_sets_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    print(f"Loading records from RecordSet: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print("No records found.")
    print()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on a numeric field, normalizing values, and grouping.

In [ ]:
# --- Choose a record set and numeric field for EDA ---
# List dataframes
for idx, k in enumerate(dataframes.keys()):
    print(f"[{idx}] {k}")
# Example: choose the main protein table record set id. Update if different!
record_set_id = record_sets_ids[0]
# Inspect columns
df = dataframes[record_set_id]
print("Available fields (columns):", df.columns.tolist())

# Choose a numeric field by @id (edit if necessary after inspecting column names)
# Example options: 'cr:column:mw', 'cr:column:Abundance1', etc.
numeric_field = None
for col in df.columns:
    # Find a likely numeric field
    if any([keyword in col.lower() for keyword in ['mw', 'abundance', 'peptide', 'coverage', 'pi']]):
        numeric_field = col
        break
if numeric_field is None:
    numeric_field = df.columns[0]
print(f"Selected numeric field for EDA: {numeric_field}\n")

threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
# Try to coerce to float
try:
    df[numeric_field] = pd.to_numeric(df[numeric_field], errors='coerce')
except:
    pass
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold:.3f}:")
display(filtered_df.head())

# Normalize the numeric field
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by another field if a textual or categorical field is present
# Find a category/group field
group_field = None
for col in df.columns:
    if col != numeric_field and df[col].dtype == object:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"\nGrouped {numeric_field} mean by {group_field}:")
    display(grouped_df.head())

## 5. Visualization
Visualize data distributions and relationships between fields using matplotlib and seaborn. You can customize field names (by `@id`) below.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Numeric field histogram
if numeric_field:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()

# Boxplot by group (if group field exists)
if group_field:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=group_field, y=numeric_field, data=filtered_df)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
In this notebook, you loaded and explored the FAIR² mass spectrometry dataset using the `mlcroissant` library. You:
- Inspected metadata and available record sets with their unique `@id` values
- Loaded tabular data programmatically
- Ran EDA including filtering and normalization using field/column `@id`s
- Produced data visualizations

Adjust analysis steps and field selections as needed to fit your scientific goals and cite the dataset appropriately if used in publications.